# Support Integrity Auditor (SIA) - Full Reproducible Pipeline
This notebook contains the complete end-to-end pipeline for the Support Integrity Auditor (SIA) project, covering:
1. **Exploratory Data Analysis & Setup**
2. **Stage 1: Self-Supervised Pseudo-Label Generation** (Fusing Rule-Based NLP & Embedding-Based Clustering)
3. **Stage 2: Sequence Classifier Training** (Fine-tuning DistilBERT on CPU)
4. **Stage 3: Evidence Dossier Generation** for priority mismatches

## 1. Environment Setup & Data Loading

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score

torch.manual_seed(42)
np.random.seed(42)

# Check if dataset exists, if not download it
if not os.path.exists('customer_support_tickets.csv'):
    print("Downloading dataset...")
    import urllib.request
    url = "https://huggingface.co/datasets/chiapudding/kaggle-customer-service/resolve/main/customer_support_tickets.csv"
    urllib.request.urlretrieve(url, 'customer_support_tickets.csv')
    print("Download complete.")

df = pd.read_csv('customer_support_tickets.csv')
print(f"Loaded dataset with shape: {df.shape}")
df.head(2)

## 2. Stage 1: Self-Supervised Pseudo-Labeling
Here we define our two independent signals:
- **Signal 1: Rule-Based NLP Urgency Score** (extracting keywords and negations)
- **Signal 2: Embedding-Based Clustering Urgency Score** (unsupervised clustering using sentence-transformers, scored by cluster averages)

Then we fuse them using a linear combination and map the score to four severity classes, yielding the binary mismatch label.

In [ ]:
def get_rule_score(text, ticket_type):
    text = str(text).lower()
    critical_keywords = [
        'data loss', 'deleted', 'disappeared', 'lost', 'locked', 'invalid credentials',
        'password reset not working', 'forgotten my password', 'reset option is not working',
        'not turning on', 'does not respond', 'no response', 'charging properly',
        'flickering', 'crashes', 'crash', 'strange noises', 'hardware problem',
        'outage', 'hacked', 'compromise', 'stolen', 'smoke', 'fire', 'original charger'
    ]
    medium_keywords = [
        'intermittent', 'firmware', 'software bug', 'software update', 'error message',
        'freezes', 'disconnects', 'internet connection', 'wi-fi network', 'wi-fi',
        'battery life', 'cannot connect', 'fail to connect', 'unresolved', 'glitch'
    ]
    low_keywords = [
        'guide me', 'steps', 'how can i', 'find the option', 'desired action', 'product inquiry'
    ]
    
    score = 0.5
    crit_count = sum(1 for kw in critical_keywords if kw in text)
    med_count = sum(1 for kw in medium_keywords if kw in text)
    low_count = sum(1 for kw in low_keywords if kw in text)
    
    if crit_count > 0:
        score = 0.85 + 0.05 * min(crit_count, 3)
    elif med_count > 0:
        score = 0.55 + 0.05 * min(med_count, 3)
    elif low_count > 0:
        score = 0.25 - 0.05 * min(low_count, 3)
        
    if '!' in text:
        score += 0.05
        
    return min(max(score, 0.0), 1.0)

print("Generating Signal 1...")
df['rule_score'] = df.apply(lambda r: get_rule_score(r['Ticket Description'], r['Ticket Type']), axis=1)

print("Generating Signal 2 (Sentence Embeddings)...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = st_model.encode(df['Ticket Description'].fillna('').tolist(), show_progress_bar=True)

print("Running K-Means Clustering...")
kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster'] = kmeans.fit_predict(embeddings)

cluster_rule_means = df.groupby('cluster')['rule_score'].mean().to_dict()
df['cluster_score'] = df['cluster'].map(cluster_rule_means)

print("Fusing signals...")
df['fused_score'] = 0.5 * df['rule_score'] + 0.5 * df['cluster_score']

def map_to_severity(score):
    if score < 0.35:
        return 'Low'
    elif score < 0.55:
        return 'Medium'
    elif score < 0.75:
        return 'High'
    else:
        return 'Critical'

df['inferred_severity'] = df['fused_score'].apply(map_to_severity)
df['mismatch'] = (df['inferred_severity'] != df['Ticket Priority']).astype(int)

print(f"Signal correlation: {df['rule_score'].corr(df['cluster_score']):.4f}")
print(df['mismatch'].value_counts(normalize=True))

## 3. Stage 2: Classifier Training
We train a `distilbert-base-uncased` sequence classifier to predict the 4 severity classes (`Low`, `Medium`, `High`, `Critical`). At inference, we compare the prediction to `Ticket Priority` to output the mismatch label. We freeze the lower transformer blocks to keep CPU training efficient.

In [ ]:
severity_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
df['severity_label'] = df['inferred_severity'].map(severity_map)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['severity_label'])

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

class SeverityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = SeverityDataset(train_df['Ticket Description'].fillna('').tolist(), train_df['severity_label'].tolist(), tokenizer)
val_dataset = SeverityDataset(val_df['Ticket Description'].fillna('').tolist(), val_df['severity_label'].tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)
device = torch.device('cpu')
model = model.to(device)

# Freeze backbone
for name, param in model.named_parameters():
    if "transformer.layer.5" not in name and "pre_classifier" not in name and "classifier" not in name:
        param.requires_grad = False

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Fine-tuning model for 1 epoch (demonstration)...")
model.train()
for step, batch in enumerate(train_loader):
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels_tensor = batch['label'].to(device)
    
    outputs = model(input_ids, attention_mask=attention_mask)
    loss = criterion(outputs.logits, labels_tensor)
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

print("Finished training demonstration.")

## 4. Evaluation & Dossier Inspection
Let's evaluate the model on the validation set, map predicted severity to mismatch, and print a sample Evidence Dossier.

In [ ]:
model.eval()
val_preds = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids, attention_mask=attention_mask).logits
        val_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())

rev_severity_map = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Critical'}
val_df = val_df.copy()
val_df['predicted_severity'] = [rev_severity_map[p] for p in val_preds]
val_df['predicted_mismatch'] = (val_df['predicted_severity'] != val_df['Ticket Priority']).astype(int)

print("Binary Mismatch Accuracy:", accuracy_score(val_df['mismatch'], val_df['predicted_mismatch']))
print(classification_report(val_df['mismatch'], val_df['predicted_mismatch']))

In [ ]:
# Sample Evidence Dossier Generation
from predict import generate_dossier
mismatch_sample = val_df[val_df['predicted_mismatch'] == 1].iloc[0]
dossier = generate_dossier(mismatch_sample, mismatch_sample['predicted_severity'], 0.99)
print(json.dumps(dossier, indent=2))